In [1]:
import pandas as pd
import re
import os
import glob
import csv
import xml.etree.ElementTree as ET
from typing import List, Dict, Tuple, Optional

In [2]:
####helper function for benchmark
def prepare_dna_mut_for_benchmark(df_cot):
    df_cot_dna=df_cot[['PMID','DNA mutation']].drop_duplicates()
    
    #####DNA#####
    df_cot_dna['DNA mutation']=df_cot_dna['DNA mutation'].apply(lambda x: x[2:] if x[:2]=='c.'else x)
    df_cot_dna=df_cot_dna[df_cot_dna['DNA mutation'].str.contains(r'\d', na=False)]
    #remove the space, _, () for better comparison
    df_cot_dna['DNA mutation']=df_cot_dna['DNA mutation'].str.replace(r"[ _()-]", "", regex=True)
    df_cot_dna['DNA mutation']=df_cot_dna['DNA mutation'].str.replace(r"incDNA", "", regex=True)
    df_cot_dna['DNA mutation']=df_cot_dna['DNA mutation'].str.replace(r"by", "", regex=True)
    df_cot_dna['DNA mutation']=df_cot_dna['DNA mutation'].str.replace(r"promoter", "", regex=True)
    return df_cot_dna

def prepare_dna_groud_truth_tmvar(df_tmVar):
    df_tmVar_dna=df_tmVar[(df_tmVar['Type']=='DNAMutation')|\
                          (df_tmVar['Type']=='SNP')|\
                          (df_tmVar['Type']=='Mutation')|\
                          (df_tmVar['Type']=='Variant')|(df_tmVar['Type']=='RS')] #for OSIRIS AND SETH
    
    #####DNA#####
    df_tmVar_dna.loc[:,'Mutation']=df_tmVar_dna['Mutation'].apply(lambda x: x[2:] if x[:2]==r'c.'else x)
    df_tmVar_dna=df_tmVar_dna[df_tmVar_dna['Mutation'].str.contains(r'\d', na=False)]

    df_tmVar_dna['Mutation']=df_tmVar_dna['Mutation'].str.replace(r"[ _()-]", "", regex=True)
    df_tmVar_dna['Mutation']=df_tmVar_dna['Mutation'].str.replace(r"promoter", "", regex=True)
    df_tmVar_dna['Mutation']=df_tmVar_dna['Mutation'].str.replace(r"by", "", regex=True)
    df_tmVar_dna['Mutation']=df_tmVar_dna['Mutation'].str.replace(r"incDNA", "", regex=True)

    return df_tmVar_dna

#######protein

def replace_amino_acids(text):
    def replace_match(match):
        return new_three_to_one_letter_map.get(match.group(0), match.group(0))

    # Replace three-letter amino acid codes found in the string
    # should allow wrong upper/lower case spelling, e.g., {R753Q, R753GLn}	{R753Q}
    return re.sub(r'[A-Za-z]{3}', replace_match, text)

def prepare_protein_mut_for_benchmark(df_cot):
    

    df_cot_pro=df_cot[['PMID','protein mutation']].drop_duplicates()

    df_cot_pro['protein mutation']=df_cot_pro['protein mutation'].astype(str)
    df_cot_pro['protein mutation']=df_cot_pro['protein mutation'].apply(lambda x: x[2:] if x[:2]=='p.'else x)
    df_cot_pro=df_cot_pro[df_cot_pro['protein mutation'].str.contains(r'\d', na=False)]

#turn 3 letter into 1 letter (not the opposite because there will be issue for words with a upper letter)
# Function to replace three-letter codes with one-letter codes in a string

    df_cot_pro['protein mutation']=df_cot_pro['protein mutation'].apply(replace_amino_acids)

#remove the space, _, () for better comparison
    df_cot_pro['protein mutation']=df_cot_pro['protein mutation'].str.replace(r"[ _()-]", "", regex=True)
#remove the 'to', 'by' and replace 'stop' as '*'
    df_cot_pro['protein mutation']=df_cot_pro['protein mutation'].str.replace(r"to", "", regex=True)
    df_cot_pro['protein mutation']=df_cot_pro['protein mutation'].str.replace(r"stop", "*", regex=True)
    df_cot_pro['protein mutation']=df_cot_pro['protein mutation'].str.replace(r"by", "", regex=True)

    return df_cot_pro

def prepare_protein_groud_truth_tmvar(df_tmVar):
    df_tmVar_pro=df_tmVar[(df_tmVar['Type']=='ProteinMutation')|(df_tmVar['Type']=='ProteinAllele')|\
                          (df_tmVar['Type']=='AcidChange')|\
                          (df_tmVar['Type']=='Mutation')|\
                          (df_tmVar['Type']=='Variant')]#for OSIRIS
    
    df_tmVar_pro.loc[:, 'Mutation']=df_tmVar_pro['Mutation'].apply(lambda x: x[2:] if x[:2]==r'p.'else x)
    df_tmVar_pro=df_tmVar_pro[df_tmVar_pro['Mutation'].str.contains(r'\d', na=False)]
    df_tmVar_pro['Mutation']=df_tmVar_pro['Mutation'].apply(replace_amino_acids)
    df_tmVar_pro['Mutation']=df_tmVar_pro['Mutation'].str.replace(r"[ _()-]", "", regex=True)
    df_tmVar_pro['Mutation']=df_tmVar_pro['Mutation'].str.replace(r"by", "", regex=True)
    df_tmVar_pro['Mutation']=df_tmVar_pro['Mutation'].str.replace(r"to", "", regex=True)
    df_tmVar_pro['Mutation']=df_tmVar_pro['Mutation'].str.replace(r"stop", "*", regex=True)
    
    return df_tmVar_pro

In [71]:
def read_pubtator_format(path):
    """Input file with PubTator format, output pd dataframe with all entities with PMID."""
    
    with open(path, 'r') as file:
        corpus = file.readlines()

    # Regular expression to identify rows starting with a PubMed ID followed by tab-separated values
    pattern = r'^\d+\t.+'

    # Filter rows matching the pattern
    table_rows = [line.strip() for line in corpus if re.match(pattern, line)]
    data = [row.split('\t') for row in table_rows]
    df = pd.DataFrame(data)
    df.columns=['PMID','start','end','Mutation','Type']
    return df


In [77]:
def load_seth_corpus_xml_labels(
    xml_path: str,
    sep: str = "\n",
    keep_doc_text: bool = False,
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    """
    Parse SETH/resources/.../corpus.xml and extract <gene> and <variant> mentions with offsets.

    Offsets are computed on doc_text = Title_text + sep + Abstract_text (if both exist).

    Returns:
      labels_df: DataFrame with columns:
        PMID, start, end, text, Type, section, source, attrs...
      docs_df (optional): DataFrame with PMID, title, abstract, doc_text
    """

    tree = ET.parse(xml_path)
    root = tree.getroot()

    label_rows: List[Dict] = []
    doc_rows: List[Dict] = []

    def _reconstruct_with_spans(node: ET.Element, pmid: str, section: str, base_offset: int) -> Tuple[str, List[Dict]]:
        """
        Reconstruct plain text from an element containing mixed content (text + child tags),
        while recording spans for child tags <gene> and <variant>.
        Offsets are relative to doc_text, so we use base_offset + running length.
        """
        parts: List[str] = []
        spans: List[Dict] = []
        cur_len = 0

        # node.text occurs before any first child
        if node.text:
            parts.append(node.text)
            cur_len += len(node.text)

        for child in list(node):
            # child element text content (the mention itself)
            mention_text = child.text or ""
            tag = child.tag  # "gene" or "variant" (expected)

            # record span if gene/variant
            if tag in ("gene", "variant") and mention_text:
                start = base_offset + cur_len
                end = start + len(mention_text)
                row = {
                    "PMID": pmid,
                    "start": start,
                    "end": end,
                    "text": mention_text,
                    "Type": "Gene" if tag == "gene" else "Variant",
                    "section": section,
                    "source": "SETH_XML",
                }
                # include attributes (g_id/g_lex/v_lex/v_norm etc.)
                for k, v in child.attrib.items():
                    row[k] = v
                spans.append(row)

            # append the mention text to reconstructed plain text
            parts.append(mention_text)
            cur_len += len(mention_text)

            # child.tail occurs after the child element before next sibling
            if child.tail:
                parts.append(child.tail)
                cur_len += len(child.tail)

        return "".join(parts), spans

    # The file structure is <Articles><Article>...
    for art in root.findall(".//Article"):
        pmid_el = art.find("Pmid")
        if pmid_el is None or pmid_el.text is None:
            continue
        pmid = pmid_el.text.strip()

        title_el = art.find("Title")
        abstract_el = art.find("Abstract")

        title_text = ""
        abstract_text = ""
        title_spans: List[Dict] = []
        abstract_spans: List[Dict] = []

        # Title part: base_offset = 0
        if title_el is not None:
            title_text, title_spans = _reconstruct_with_spans(
                node=title_el,
                pmid=pmid,
                section="Title",
                base_offset=0
            )

        # Abstract part: base_offset = len(title_text) + len(sep) (if title exists), else 0
        if abstract_el is not None:
            base = len(title_text) + (len(sep) if title_text else 0)
            abstract_text, abstract_spans = _reconstruct_with_spans(
                node=abstract_el,
                pmid=pmid,
                section="Abstract",
                base_offset=base
            )

        # finalize doc_text (used for offset reference)
        doc_text = title_text + (sep + abstract_text if title_text and abstract_text else abstract_text)

        # collect spans
        label_rows.extend(title_spans)
        label_rows.extend(abstract_spans)

        if keep_doc_text:
            doc_rows.append({
                "PMID": pmid,
                "title": title_text,
                "abstract": abstract_text,
                "doc_text": doc_text
            })

    labels_df = pd.DataFrame(label_rows).sort_values(["PMID", "start", "end"]).reset_index(drop=True)
    docs_df = pd.DataFrame(doc_rows) if keep_doc_text else None
    return labels_df, docs_df


In [70]:
pm_tv3 = pd.read_csv('results/PubMind/Wei2013_test.pubmind.out.raw.tsv',sep='\t',index_col=0)
pm_tv3.head()

,PMID,gene,DNA mutation,protein mutation,disease,phenotype,LLM reasoning,pathogenicity
0,21738389,DFNB31,c.737delC,p.Pro246HisfsX13,Usher syndrome,severe rod-cone dystrophy and varying degrees of hearing impairment,predicted to lead to a truncated protein,pathogenic
1,22051099,CXCR1,rs2234671,S276T,chronic periodontitis,-,not associated with the susceptibility to chronic periodontitis,benign
2,22188495,WRN,3190C>T,Arg987Ter,Werner syndrome,"cataracts, hair graying, tendinous calcinosis",the mutated WRN protein was unable to translocate into the nucleus in an in vitro cell assay,pathogenic
3,21881116,HBV,insertion/deletion in precore region,frameshift of precore protein,fulminant hepatitis (FH),-,enhanced HBV replication,pathogenic
4,21881116,HBV,insertion of A between nucleotides 1900 and 1901,3-nucleotide change within the Kozak sequence of the core protein,fulminant hepatitis (FH),-,enhanced core protein expression,pathogenic


In [5]:
seth_tv3 = pd.read_csv('results/SETH/tmVar3.seth.out.tsv',sep='\t')
seth_tv3.head()

,PMID,start,end,text,type,tool
0,12636044,477,496,Ala1492 with an Asp,SUBSTITUTION,MUTATIONFINDER
1,12650796,990,998,3020insC,INSERTION,REGEX
2,12650796,295,303,3020insC,INSERTION,REGEX
3,12650796,63,71,3020insC,INSERTION,REGEX
4,12650796,1362,1370,3020insC,INSERTION,REGEX


In [106]:
pm_tv3 = pd.read_csv('results/PubMind/tmVar3.pubmind.out.raw.tsv',sep='\t',index_col=0)
pm_tv3.head()

,PMID,gene,DNA mutation,protein mutation,disease,phenotype,LLM reasoning,pathogenicity
0,22051099,CXCR1,rs2234671,S276T,chronic periodontitis,-,not associated with the susceptibility to chronic periodontitis,benign
1,22188495,WRN,3190C>T,Arg987Ter,Werner syndrome,"cataracts, hair graying, tendinous calcinosis",the mutated WRN protein was unable to translocate into the nucleus in an in vitro cell assay,pathogenic
2,20846357,GJB2,c.263C>T,p.Ala88Val,Keratitis-ichthyosis-deafness (KID) syndrome,"hearing impairment, ichthyosiform erythroderma, palmoplantar keratoderma, alopecia, thick vernix...",novel nucleotide change leading to a substitution of alanine for valine,pathogenic
3,21099701,CTSK,c.908G>A,p.Gly303Glu,pycnodysostosis,"short stature, osteosclerosis, delayed cranial suture closure, hypoplastic mandible, acro-osteol...",novel homozygous mutation leading to amino acid substitution,pathogenic
4,20801104,DRD4,-120 bp duplication,-,schizophrenia,-,strong association with schizophrenia (p=0.008),likely pathogenic


In [6]:
gt_tv3 = read_pubtator_format('corpus_groudtruth/tmVar3Corpus.txt')[[0,3,4]]
gt_tv3.columns = ['PMID','Mutation','Type']
gt_tv3.head()

,PMID,Mutation,Type
0,22051099,CXCR1,Gene
1,22051099,IL8RA,Gene
2,22051099,chemokine receptor 1 CXCR-1,Gene
3,22051099,IL8R-alpha,Gene
4,22051099,interleukin 8,Gene


In [71]:
#try Wei2013 corpus
gt_tv3 = read_pubtator_format('corpus_groudtruth/Wei2013/test.txt')[[0,3,4]]
gt_tv3.columns = ['PMID','Mutation','Type']
gt_tv3.head()

,PMID,Mutation,Type
0,21738389,c.737delC,DNAMutation
1,21738389,p.Pro246HisfsX13,ProteinMutation
2,22051099,rs2234671,SNP
3,22051099,Ex2+860G>C,DNAMutation
4,22051099,S276T,ProteinMutation


In [43]:
#try Wei2013 corpus
gt_wei13 = read_pubtator_format('corpus_groudtruth/Wei2013/test.txt')[[0,3,4]]
gt_wei13.columns = ['PMID','Mutation','Type']
gt_wei13.head()

,PMID,Mutation,Type
0,21738389,c.737delC,DNAMutation
1,21738389,p.Pro246HisfsX13,ProteinMutation
2,22051099,rs2234671,SNP
3,22051099,Ex2+860G>C,DNAMutation
4,22051099,S276T,ProteinMutation


In [72]:
pm_tv3[pm_tv3['PMID']==16480574]

,PMID,gene,DNA mutation,protein mutation,disease,phenotype,LLM reasoning,pathogenicity
233,16480574,LMP1,-,p.(346-355del; 334X; 335X; 338X; 366X),nasopharyngeal carcinoma (NPC),-,30-bp deletion and 4 missense point mutations,pathogenic
234,16480574,LMP1,-,p.(346-355del),nasopharyngeal carcinoma (NPC),-,30-bp deletion,pathogenic
235,16480574,LMP1,-,p.(334X; 335X; 338X; 366X),nasopharyngeal carcinoma (NPC),-,4 missense point mutations,pathogenic
236,16480574,LMP1,-,p.(no mutation),nasopharyngeal carcinoma (NPC),-,no mutation,benign


In [73]:
gt_tv3[gt_tv3['PMID']=='16480574']

,PMID,Mutation,Type


In [7]:
def clean_dna_mut_for_benchmark(df):
    df_dna = df.copy()
    df_dna['Mutation']=df_dna['Mutation'].apply(lambda x: x[2:] if x[:2]=='c.'else x)
    df_dna=df_dna[df_dna['Mutation'].str.contains(r'\d', na=False)]
    #remove the space, _, () for better comparison
    df_dna['Mutation']=df_dna['Mutation'].str.replace(r"[ _()-]", "", regex=True)
    df_dna['Mutation']=df_dna['Mutation'].str.replace(r"incDNA", "", regex=True)
    df_dna['Mutation']=df_dna['Mutation'].str.replace(r"by", "", regex=True)
    df_dna['Mutation']=df_dna['Mutation'].str.replace(r"promoter", "", regex=True)
    return df_dna

def dna_benchmark_tool_with_gt(
    df_tool_dna: pd.DataFrame,
    df_gt_dna: pd.DataFrame,
    tool: str,
    gt_col: str = "Mutation",
    tool_col: str = "Mutation",
    match_mode: str = "substring",   # "exact" or "substring"
    macro: bool = True,
    verbose: bool = True
):
    """
    Compare tool vs GT for DNA variants using per-PMID sets.

    Returns:
      merged_df: per-PMID sets + TP/FP/FN + per-PMID metrics
      summary: dict of micro + macro metrics

    Metrics:
      precision = TP / (TP + FP)
      recall    = TP / (TP + FN)
      f1        = 2PR/(P+R)
      accuracy  = Jaccard = TP / (TP + FP + FN)   (set-based accuracy)
    """

    tool_mut_col = f"{tool}_Mutations"
    tool_total_col = f"Total_{tool}"

    # ---- group into sets per PMID ----
    gt_grouped = df_gt_dna.groupby("PMID")[gt_col].apply(lambda x: set(x.dropna())).reset_index(name="GT_Mutations")
    tool_grouped = df_tool_dna.groupby("PMID")[tool_col].apply(lambda x: set(x.dropna())).reset_index(name=tool_mut_col)

    gt_grouped["PMID"] = gt_grouped["PMID"].astype(str)
    tool_grouped["PMID"] = tool_grouped["PMID"].astype(str)

    merged = pd.merge(gt_grouped, tool_grouped, on="PMID", how="outer")
    merged["GT_Mutations"] = merged["GT_Mutations"].apply(lambda x: x if isinstance(x, set) else set())
    merged[tool_mut_col] = merged[tool_mut_col].apply(lambda x: x if isinstance(x, set) else set())

    # ---- matching functions ----
    def _hit(gt_item: str, tool_item: str) -> bool:
        if match_mode == "exact":
            return gt_item == tool_item
        # substring mode: treat GT as matched if GT string occurs in tool string OR vice versa
        # (robust when one has extra context)
        return (gt_item in tool_item) or (tool_item in gt_item)

    def _count_tp_fp_fn(gt_set: set, tool_set: set):
        # TP: how many GT items are matched by any tool item
        tp = 0
        for g in gt_set:
            if any(_hit(str(g), str(t)) for t in tool_set):
                tp += 1

        fn = len(gt_set) - tp

        # FP: tool items that do not match any GT item
        fp = 0
        for t in tool_set:
            if not any(_hit(str(g), str(t)) for g in gt_set):
                fp += 1

        return tp, fp, fn

    # ---- compute TP/FP/FN per PMID ----
    merged[["TP", "FP", "FN"]] = merged.apply(
        lambda row: pd.Series(_count_tp_fp_fn(row["GT_Mutations"], row[tool_mut_col])),
        axis=1
    )

    merged["Total_GT"] = merged["GT_Mutations"].apply(len)
    merged[tool_total_col] = merged[tool_mut_col].apply(len)

    # per-PMID metrics
    def _prf(tp, fp, fn):
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2*p*r/(p+r)) if (p+r) else 0.0
        acc = tp / (tp + fp + fn) if (tp + fp + fn) else 0.0  # Jaccard accuracy
        return p, r, f1, acc

    merged[["Precision", "Recall", "F1", "Accuracy"]] = merged.apply(
        lambda row: pd.Series(_prf(row["TP"], row["FP"], row["FN"])),
        axis=1
    )

    # ---- micro aggregation ----
    TP = int(merged["TP"].sum())
    FP = int(merged["FP"].sum())
    FN = int(merged["FN"].sum())

    micro_p, micro_r, micro_f1, micro_acc = _prf(TP, FP, FN)

    summary = {
        "tool": tool,
        "match_mode": match_mode,
        "micro": {
            "TP": TP, "FP": FP, "FN": FN,
            "precision": micro_p,
            "recall": micro_r,
            "f1": micro_f1,
            "accuracy_jaccard": micro_acc,
        }
    }

    # ---- macro aggregation (optional) ----
    if macro:
        df_nonempty = merged[merged["Total_GT"] > 0].copy()
        summary["macro"] = {
            "precision": float(df_nonempty["Precision"].mean()) if len(df_nonempty) else 0.0,
            "recall": float(df_nonempty["Recall"].mean()) if len(df_nonempty) else 0.0,
            "f1": float(df_nonempty["F1"].mean()) if len(df_nonempty) else 0.0,
            "accuracy_jaccard": float(df_nonempty["Accuracy"].mean()) if len(df_nonempty) else 0.0,
        }

    if verbose:
        m = summary["micro"]
        print(
            f"## {tool} DNA benchmark ({match_mode}) | "
            f"TP={m['TP']} FP={m['FP']} FN={m['FN']} | "
            f"P={m['precision']:.4f} R={m['recall']:.4f} F1={m['f1']:.4f} Acc(Jacc)={m['accuracy_jaccard']:.4f}"
        )

    # return per-PMID details + summary
    keep_cols = ["PMID", "GT_Mutations", tool_mut_col, "TP", "FP", "FN", "Total_GT", tool_total_col,
                 "Precision", "Recall", "F1", "Accuracy"]
    return merged[keep_cols], summary


In [8]:
gt_tv3_dna = gt_tv3[(gt_tv3['Type']=='SNP')|\
        (gt_tv3['Type']=='DNAMutation')|\
        (gt_tv3['Type']=='OtherMutation')|\
        (gt_tv3['Type']=='DNAAllele')]
gt_tv3_dna

,PMID,Mutation,Type
6,22051099,rs2234671,SNP
7,22051099,Ex2+860G>C,DNAMutation
12,22051099,rs2234671,SNP
13,22051099,rs2234671,SNP
22,22188495,3190C>T,DNAMutation
...,...,...,...
7813,12650796,cytosine insertion at position 3020,DNAMutation
7814,12650796,3020insC,DNAMutation
7820,12650796,3020insC,DNAMutation
7822,12650796,3020insC,DNAMutation


In [110]:
pm_tv3_dna = pm_tv3[pm_tv3['DNA mutation']!='-'][['PMID','DNA mutation']].drop_duplicates().rename(columns={'DNA mutation':'Mutation'})

In [111]:
result, summary = dna_benchmark_tool_with_gt(pm_tv3_dna, gt_tv3_dna, tool="PubMind", match_mode="substring")

## PubMind DNA benchmark (substring) | TP=501 FP=305 FN=310 | P=0.6216 R=0.6178 F1=0.6197 Acc(Jacc)=0.4489


In [112]:
result2, summary2 = dna_benchmark_tool_with_gt(clean_dna_mut_for_benchmark(pm_tv3_dna), 
                        clean_dna_mut_for_benchmark(gt_tv3_dna), 
                        tool="PubMind", 
                        match_mode="substring")

## PubMind DNA benchmark (substring) | TP=524 FP=230 FN=234 | P=0.6950 R=0.6913 F1=0.6931 Acc(Jacc)=0.5304


In [113]:
summary

{'tool': 'PubMind',
 'match_mode': 'substring',
 'micro': {'TP': 501,
  'FP': 305,
  'FN': 310,
  'precision': 0.6215880893300249,
  'recall': 0.6177558569667078,
  'f1': 0.6196660482374768,
  'accuracy_jaccard': 0.4489247311827957},
 'macro': {'precision': 0.7054053724053724,
  'recall': 0.6213455433455434,
  'f1': 0.6304682726180703,
  'accuracy_jaccard': 0.5616606857195092}}

In [114]:
summary2

{'tool': 'PubMind',
 'match_mode': 'substring',
 'micro': {'TP': 524,
  'FP': 230,
  'FN': 234,
  'precision': 0.6949602122015915,
  'recall': 0.6912928759894459,
  'f1': 0.693121693121693,
  'accuracy_jaccard': 0.5303643724696356},
 'macro': {'precision': 0.7515686420858834,
  'recall': 0.6841717669303876,
  'f1': 0.688728553614711,
  'accuracy_jaccard': 0.6293464198636612}}

In [115]:
tmv3_tv3 = read_pubtator_format('results/tmVar3/tmVar3Corpus_text_only.bioc.xml.PubTator')[[0,3,4]]
tmv3_tv3.columns = ['PMID','Mutation','Type']
tmv3_tv3.head()


,PMID,Mutation,Type
0,22051099,rs2234671,SNP
1,22051099,Ex2+860G>C,DNAMutation
2,22051099,S276T,ProteinMutation
3,22051099,rs2234671,SNP
4,22051099,rs2234671,SNP


In [81]:
tmv3_tv3 = read_pubtator_format('results/tmVar3/Wei2013_test_text_only.bioc.xml.PubTator')[[0,3,4]]
tmv3_tv3.columns = ['PMID','Mutation','Type']
tmv3_tv3.head()


,PMID,Mutation,Type
0,21738389,c.737delC,DNAMutation
1,21738389,p.Pro246HisfsX13,ProteinMutation
2,21738389,chromosome 2,Chromosome
3,21738389,chromosome 9,Chromosome
4,22051099,rs2234671,SNP


In [116]:
tmv3_tv3_dna = tmv3_tv3[(tmv3_tv3['Type']=='SNP')|\
        (tmv3_tv3['Type']=='DNAMutation')|\
        (tmv3_tv3['Type']=='OtherMutation')|\
        (tmv3_tv3['Type']=='DNAAllele')]
tmv3_tv3_dna

,PMID,Mutation,Type
0,22051099,rs2234671,SNP
1,22051099,Ex2+860G>C,DNAMutation
3,22051099,rs2234671,SNP
4,22051099,rs2234671,SNP
5,22188495,3190C>T,DNAMutation
...,...,...,...
1893,12650796,3020insC,DNAMutation
1894,12650796,3020insC,DNAMutation
1895,12650796,3020insC,DNAMutation
1896,12650796,3020insC,DNAMutation


In [117]:
result_tv3, summary_tv3 = dna_benchmark_tool_with_gt(tmv3_tv3_dna, 
                        gt_tv3_dna, 
                        tool="tmVar3", 
                        match_mode="substring")

## tmVar3 DNA benchmark (substring) | TP=603 FP=14 FN=208 | P=0.9773 R=0.7435 F1=0.8445 Acc(Jacc)=0.7309


In [118]:
result2_tv3, summary2_tv3 = dna_benchmark_tool_with_gt(clean_dna_mut_for_benchmark(tmv3_tv3_dna), 
                        clean_dna_mut_for_benchmark(gt_tv3_dna), 
                        tool="tmVar3", 
                        match_mode="substring")

## tmVar3 DNA benchmark (substring) | TP=594 FP=14 FN=164 | P=0.9770 R=0.7836 F1=0.8697 Acc(Jacc)=0.7694


In [9]:
def clean_pro_mut_for_benchmark(df):
    df_tmVar_pro = df.copy()
    df_tmVar_pro.loc[:, 'Mutation']=df_tmVar_pro['Mutation'].apply(lambda x: x[2:] if x[:2]==r'p.'else x)
    df_tmVar_pro=df_tmVar_pro[df_tmVar_pro['Mutation'].str.contains(r'\d', na=False)]
    #df_tmVar_pro['Mutation']=df_tmVar_pro['Mutation'].apply(replace_amino_acids)
    df_tmVar_pro['Mutation']=df_tmVar_pro['Mutation'].str.replace(r"[ _()-]", "", regex=True)
    df_tmVar_pro['Mutation']=df_tmVar_pro['Mutation'].str.replace(r"by", "", regex=True)
    df_tmVar_pro['Mutation']=df_tmVar_pro['Mutation'].str.replace(r"to", "", regex=True)
    df_tmVar_pro['Mutation']=df_tmVar_pro['Mutation'].str.replace(r"stop", "*", regex=True)
    return df_tmVar_pro

def protein_benchmark_tool_with_gt(
    df_tool_pro: pd.DataFrame,
    df_gt_pro: pd.DataFrame,
    tool: str,
    gt_col: str = "Mutation",
    tool_col: str = "Mutation",
    match_mode: str = "substring",   # "exact" or "substring"
    macro: bool = True,
    verbose: bool = True
):
    """
    Compare tool vs GT for protein variants using per-PMID sets.

    Returns:
      detail_df: per-PMID sets + TP/FP/FN + per-PMID metrics
      summary: dict of micro + (optional) macro metrics

    Metrics:
      precision = TP / (TP + FP)
      recall    = TP / (TP + FN)
      f1        = 2PR/(P+R)
      accuracy  = Jaccard = TP / (TP + FP + FN)   (set-based accuracy)
    """

    tool_mut_col = f"{tool}_Mutations"
    tool_total_col = f"Total_{tool}"

    # ---- group into sets per PMID ----
    gt_grouped = df_gt_pro.groupby("PMID")[gt_col].apply(lambda x: set(x.dropna())).reset_index(name="GT_Mutations")
    tool_grouped = df_tool_pro.groupby("PMID")[tool_col].apply(lambda x: set(x.dropna())).reset_index(name=tool_mut_col)

    gt_grouped["PMID"] = gt_grouped["PMID"].astype(str)
    tool_grouped["PMID"] = tool_grouped["PMID"].astype(str)

    merged = pd.merge(gt_grouped, tool_grouped, on="PMID", how="outer")
    merged["GT_Mutations"] = merged["GT_Mutations"].apply(lambda x: x if isinstance(x, set) else set())
    merged[tool_mut_col] = merged[tool_mut_col].apply(lambda x: x if isinstance(x, set) else set())

    # ---- matching ----
    def _hit(gt_item: str, tool_item: str) -> bool:
        if match_mode == "exact":
            return gt_item == tool_item
        # substring (relaxed): treat as match if either contains the other
        # robust when tool adds context or formatting differences remain
        return (gt_item in tool_item) or (tool_item in gt_item)

    def _count_tp_fp_fn(gt_set: set, tool_set: set):
        # TP: #GT variants matched by any tool variant
        tp = 0
        for g in gt_set:
            if any(_hit(str(g), str(t)) for t in tool_set):
                tp += 1
        fn = len(gt_set) - tp

        # FP: #tool variants that match NO GT variant
        fp = 0
        for t in tool_set:
            if not any(_hit(str(g), str(t)) for g in gt_set):
                fp += 1

        return tp, fp, fn

    merged[["TP", "FP", "FN"]] = merged.apply(
        lambda row: pd.Series(_count_tp_fp_fn(row["GT_Mutations"], row[tool_mut_col])),
        axis=1
    )

    merged["Total_GT"] = merged["GT_Mutations"].apply(len)
    merged[tool_total_col] = merged[tool_mut_col].apply(len)

    # ---- per-PMID metrics ----
    def _prf(tp, fp, fn):
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2*p*r/(p+r)) if (p+r) else 0.0
        acc = tp / (tp + fp + fn) if (tp + fp + fn) else 0.0  # Jaccard
        return p, r, f1, acc

    merged[["Precision", "Recall", "F1", "Accuracy"]] = merged.apply(
        lambda row: pd.Series(_prf(row["TP"], row["FP"], row["FN"])),
        axis=1
    )

    # ---- micro ----
    TP = int(merged["TP"].sum())
    FP = int(merged["FP"].sum())
    FN = int(merged["FN"].sum())
    micro_p, micro_r, micro_f1, micro_acc = _prf(TP, FP, FN)

    summary = {
        "tool": tool,
        "match_mode": match_mode,
        "micro": {
            "TP": TP, "FP": FP, "FN": FN,
            "precision": micro_p,
            "recall": micro_r,
            "f1": micro_f1,
            "accuracy_jaccard": micro_acc,
        }
    }

    # ---- macro ----
    if macro:
        df_nonempty = merged[merged["Total_GT"] > 0].copy()
        summary["macro"] = {
            "precision": float(df_nonempty["Precision"].mean()) if len(df_nonempty) else 0.0,
            "recall": float(df_nonempty["Recall"].mean()) if len(df_nonempty) else 0.0,
            "f1": float(df_nonempty["F1"].mean()) if len(df_nonempty) else 0.0,
            "accuracy_jaccard": float(df_nonempty["Accuracy"].mean()) if len(df_nonempty) else 0.0,
        }

    if verbose:
        m = summary["micro"]
        print(
            f"## {tool} Protein benchmark ({match_mode}) | "
            f"TP={m['TP']} FP={m['FP']} FN={m['FN']} | "
            f"P={m['precision']:.4f} R={m['recall']:.4f} F1={m['f1']:.4f} Acc(Jacc)={m['accuracy_jaccard']:.4f}"
        )

    keep_cols = ["PMID", "GT_Mutations", tool_mut_col, "TP", "FP", "FN",
                 "Total_GT", tool_total_col, "Precision", "Recall", "F1", "Accuracy"]
    return merged[keep_cols], summary


In [10]:
gt_tv3['Type'].unique()

array(['Gene', 'SNP', 'DNAMutation', 'ProteinMutation', 'Species',
       'OtherMutation', 'DNAAllele', 'ProteinAllele', 'CellLine',
       'AcidChange'], dtype=object)

In [11]:
gt_tv3_pro = gt_tv3[(gt_tv3['Type']=='ProteinMutation')|\
        (gt_tv3['Type']=='ProteinAllele')|\
        (gt_tv3['Type']=='AcidChange')]
gt_tv3_pro

,PMID,Mutation,Type
9,22051099,S276T,ProteinMutation
23,22188495,arginine residue at position 573 by terminatio...,ProteinMutation
24,22188495,Arg987Ter,ProteinMutation
27,22188495,Arg987Ter,ProteinMutation
33,20846357,p.Asp50Asn,ProteinMutation
...,...,...,...
7376,15820770,threonine for lysine,AcidChange
7430,15667541,Glu29 to Lys,ProteinMutation
7431,15667541,E29K,ProteinMutation
7432,15667541,E29K,ProteinMutation


In [122]:
pm_tv3_pro = pm_tv3[pm_tv3['protein mutation']!='-'][['PMID','protein mutation']].drop_duplicates().rename(columns={'protein mutation':'Mutation'})

In [123]:
result_pro, summary_pro = protein_benchmark_tool_with_gt(pm_tv3_pro,gt_tv3_pro,'PubMind',match_mode='substring')

## PubMind Protein benchmark (substring) | TP=317 FP=153 FN=269 | P=0.6745 R=0.5410 F1=0.6004 Acc(Jacc)=0.4290


In [124]:
result_pro2, summary_pro2 = protein_benchmark_tool_with_gt(clean_pro_mut_for_benchmark(pm_tv3_pro),
                                clean_pro_mut_for_benchmark(gt_tv3_pro),
                                'PubMind',
                                match_mode='substring')

## PubMind Protein benchmark (substring) | TP=326 FP=105 FN=179 | P=0.7564 R=0.6455 F1=0.6966 Acc(Jacc)=0.5344


In [125]:
tmv3_tv3_pro = tmv3_tv3[(tmv3_tv3['Type']=='ProteinMutation')|\
        (tmv3_tv3['Type']=='ProteinAllele')|\
        (tmv3_tv3['Type']=='AcidChange')]
tmv3_tv3_pro

,PMID,Mutation,Type
2,22051099,S276T,ProteinMutation
6,22188495,Arg987Ter,ProteinMutation
7,22188495,Arg987Ter,ProteinMutation
8,22188495,arginine residue at position 573 by termination codon,ProteinMutation
9,20846357,p.Asp50Asn,ProteinMutation
...,...,...,...
1817,15820770,ACG-->AAG substitution in codon 420,ProteinMutation
1818,15820770,GAT-->GAG substitution replaces aspartic acid by glutamic acid in codon 416,ProteinMutation
1825,15667541,Glu29 to Lys,ProteinMutation
1826,15667541,E29K,ProteinMutation


In [126]:
result_pro_tv3, summary_pro_tv3 = protein_benchmark_tool_with_gt(tmv3_tv3_pro,gt_tv3_pro,'tmVar3',match_mode='substring')

## tmVar3 Protein benchmark (substring) | TP=479 FP=7 FN=107 | P=0.9856 R=0.8174 F1=0.8937 Acc(Jacc)=0.8078


In [127]:
result_pro2_tv3, summary_pro2_tv3 = protein_benchmark_tool_with_gt(clean_pro_mut_for_benchmark(tmv3_tv3_pro),
                                clean_pro_mut_for_benchmark(gt_tv3_pro),
                                'tmVar3',
                                match_mode='substring')

## tmVar3 Protein benchmark (substring) | TP=474 FP=7 FN=31 | P=0.9854 R=0.9386 F1=0.9615 Acc(Jacc)=0.9258


In [28]:
pm_tv3[pm_tv3['PMID']==12668354]

,PMID,gene,DNA mutation,protein mutation,disease,phenotype,LLM reasoning,pathogenicity
1001,12668354,SLC6A4,5-HTTLPR,-,shyness,-,association with shyness scores,likely pathogenic
1002,12668354,DRD4,-,-,shyness,-,no significant association,benign
1003,12668354,COMT,-,-,shyness,-,no significant association,benign
1004,12668354,MAOA,-,-,shyness,-,no significant association,benign


In [36]:
pd.set_option('display.max_colwidth', 100)

In [37]:
result[result['Accuracy']==0].head(20)

,PMID,GT_Mutations,PubMind_Mutations,TP,FP,FN,Total_GT,Total_PubMind,Precision,Recall,F1,Accuracy
1,12636044,"{c4475C --> A, c2403T --> C, A insertion, c5398T --> C}","{c2403T>C, c5398T>C, c4475C>A}",0,3,4,4,3,0.0,0.0,0.0,0.0
4,12668354,{44 base pair insertion/deletion},{5-HTTLPR},0,1,1,1,1,0.0,0.0,0.0,0.0
6,12673366,{G to C substitution at position 135},{G135C},0,1,1,1,1,0.0,0.0,0.0,0.0
8,12716337,{G to C substitution at position -174},{-174G>C},0,1,1,1,1,0.0,0.0,0.0,0.0
9,12719097,{C deletion at nucleotide position 4548},{c.4548delC},0,1,1,1,1,0.0,0.0,0.0,0.0
11,12736721,{192 bp insert},{},0,0,1,1,0,0.0,0.0,0.0,0.0
26,14510914,"{15 nt duplicating, deletion of the coding sequence (nt 1314 through nt 1328)}",{c.1314_1328delins15nt},0,1,2,2,1,0.0,0.0,0.0,0.0
28,14517957,{},{missense mutation},0,1,0,0,1,0.0,0.0,0.0,0.0
31,14568816,"{stop codon at nucleotides 766 to 768, G-to-A mutation at codon 456, GATT duplication (nucleotid...","{c.1367G>A, c.763_766dupGATT}",0,2,3,3,2,0.0,0.0,0.0,0.0
33,14587045,{},"{beta(0)-thalassemia mutation, Hb E mutation, missense mutation}",0,3,0,0,3,0.0,0.0,0.0,0.0


In [39]:
result_tv3[result_tv3['Accuracy']==1].head(20)

,PMID,GT_Mutations,tmVar3_Mutations,TP,FP,FN,Total_GT,Total_tmVar3,Precision,Recall,F1,Accuracy
2,12650796,"{3020insC, cytosine insertion at position 3020}","{3020insC, cytosine insertion at position 3020}",2,0,0,2,2,1.0,1.0,1.0,1.0
5,12668609,{A-262T},{A-262T},1,0,0,1,1,1.0,1.0,1.0,1.0
6,12673366,{G to C substitution at position 135},{G to C substitution at position 135},1,0,0,1,1,1.0,1.0,1.0,1.0
8,12716337,{G to C substitution at position -174},{G to C substitution at position -174},1,0,0,1,1,1.0,1.0,1.0,1.0
9,12719097,{C deletion at nucleotide position 4548},{C deletion at nucleotide position 4548},1,0,0,1,1,1.0,1.0,1.0,1.0
16,12809638,"{2566T-->C, 2590_2591insCCCC, 752_758delGCCGGCC}","{2566T-->C, 2590_2591insCCCC, 752_758delGCCGGCC}",3,0,0,3,3,1.0,1.0,1.0,1.0
18,12829659,"{G-308G, C-174G, -308A, G-308A, C-174C}","{G-308G, C-174G, -308A, G-308A, C-174C}",5,0,0,5,5,1.0,1.0,1.0,1.0
19,12862311,{79-1 G > T},{79-1 G > T},1,0,0,1,1,1.0,1.0,1.0,1.0
21,12898858,{A1166C},{A1166C},1,0,0,1,1,1.0,1.0,1.0,1.0
24,12968672,{G-1739A},{G-1739A},1,0,0,1,1,1.0,1.0,1.0,1.0


# PubMind vs. tmVar3 (DNA and protein seperately)

## Wei2013 test corpus

In [97]:
summary2

{'tool': 'PubMind',
 'match_mode': 'substring',
 'micro': {'TP': 152,
  'FP': 90,
  'FN': 30,
  'precision': 0.628099173553719,
  'recall': 0.8351648351648352,
  'f1': 0.7169811320754716,
  'accuracy_jaccard': 0.5588235294117647},
 'macro': {'precision': 0.7799854041233353,
  'recall': 0.8275862068965517,
  'f1': 0.7764220453875628,
  'accuracy_jaccard': 0.7311348294106915}}

In [98]:
summary2_tv3

{'tool': 'tmVar3',
 'match_mode': 'substring',
 'micro': {'TP': 173,
  'FP': 38,
  'FN': 9,
  'precision': 0.8199052132701422,
  'recall': 0.9505494505494505,
  'f1': 0.8804071246819339,
  'accuracy_jaccard': 0.7863636363636364},
 'macro': {'precision': 0.8433908045977013,
  'recall': 0.9243295019157088,
  'f1': 0.8707464948844259,
  'accuracy_jaccard': 0.8393678160919541}}

In [99]:
summary_pro2

{'tool': 'PubMind',
 'match_mode': 'substring',
 'micro': {'TP': 113,
  'FP': 30,
  'FN': 24,
  'precision': 0.7902097902097902,
  'recall': 0.8248175182481752,
  'f1': 0.8071428571428572,
  'accuracy_jaccard': 0.6766467065868264},
 'macro': {'precision': 0.8604289940828402,
  'recall': 0.7868589743589743,
  'f1': 0.8074619824619824,
  'accuracy_jaccard': 0.7706854043392505}}

In [100]:
summary_pro2_tv3

{'tool': 'tmVar3',
 'match_mode': 'substring',
 'micro': {'TP': 134,
  'FP': 18,
  'FN': 3,
  'precision': 0.881578947368421,
  'recall': 0.9781021897810219,
  'f1': 0.9273356401384083,
  'accuracy_jaccard': 0.864516129032258},
 'macro': {'precision': 0.8466117216117216,
  'recall': 0.9551282051282051,
  'f1': 0.8812130177514792,
  'accuracy_jaccard': 0.8402014652014652}}

## tmVar3 corpus

In [128]:
summary2

{'tool': 'PubMind',
 'match_mode': 'substring',
 'micro': {'TP': 524,
  'FP': 230,
  'FN': 234,
  'precision': 0.6949602122015915,
  'recall': 0.6912928759894459,
  'f1': 0.693121693121693,
  'accuracy_jaccard': 0.5303643724696356},
 'macro': {'precision': 0.7515686420858834,
  'recall': 0.6841717669303876,
  'f1': 0.688728553614711,
  'accuracy_jaccard': 0.6293464198636612}}

In [129]:
summary2_tv3

{'tool': 'tmVar3',
 'match_mode': 'substring',
 'micro': {'TP': 594,
  'FP': 14,
  'FN': 164,
  'precision': 0.9769736842105263,
  'recall': 0.783641160949868,
  'f1': 0.8696925329428988,
  'accuracy_jaccard': 0.7694300518134715},
 'macro': {'precision': 0.812173458725183,
  'recall': 0.7473390420098883,
  'f1': 0.7671217708383969,
  'accuracy_jaccard': 0.7420011806375443}}

In [130]:
summary_pro2

{'tool': 'PubMind',
 'match_mode': 'substring',
 'micro': {'TP': 326,
  'FP': 105,
  'FN': 179,
  'precision': 0.7563805104408353,
  'recall': 0.6455445544554456,
  'f1': 0.6965811965811967,
  'accuracy_jaccard': 0.5344262295081967},
 'macro': {'precision': 0.7231511158594492,
  'recall': 0.599647266313933,
  'f1': 0.6362441213286624,
  'accuracy_jaccard': 0.5840307454890787}}

In [131]:
summary_pro2_tv3

{'tool': 'tmVar3',
 'match_mode': 'substring',
 'micro': {'TP': 474,
  'FP': 7,
  'FN': 31,
  'precision': 0.9854469854469855,
  'recall': 0.9386138613861386,
  'f1': 0.9614604462474645,
  'accuracy_jaccard': 0.92578125},
 'macro': {'precision': 0.9328703703703703,
  'recall': 0.917361111111111,
  'f1': 0.9222756897565326,
  'accuracy_jaccard': 0.9150462962962962}}

# combined DNA protein variant

In [15]:
gt_tv3_dnaNpro = pd.concat([gt_tv3_dna[['PMID','Mutation']],gt_tv3_pro[['PMID','Mutation']]],ignore_index=True)

In [16]:
gt_tv3_dnaNpro

,PMID,Mutation
0,22051099,rs2234671
1,22051099,Ex2+860G>C
2,22051099,rs2234671
3,22051099,rs2234671
4,22188495,3190C>T
...,...,...
1970,15820770,threonine for lysine
1971,15667541,Glu29 to Lys
1972,15667541,E29K
1973,15667541,E29K


In [17]:
def clean_dnaNpro_mut_for_benchmark(df):
    df_dna = df.copy()
    df_dna['Mutation']=df_dna['Mutation'].apply(lambda x: x[2:] if x[:2]=='c.'else x)
    df_dna.loc[:, 'Mutation']=df_dna['Mutation'].apply(lambda x: x[2:] if x[:2]==r'p.'else x)
    df_dna=df_dna[df_dna['Mutation'].str.contains(r'\d', na=False)]
    #remove the space, _, () for better comparison
    df_dna['Mutation']=df_dna['Mutation'].str.replace(r"[ _()-]", "", regex=True)
    df_dna['Mutation']=df_dna['Mutation'].str.replace(r"incDNA", "", regex=True)
    df_dna['Mutation']=df_dna['Mutation'].str.replace(r"by", "", regex=True)
    df_dna['Mutation']=df_dna['Mutation'].str.replace(r"promoter", "", regex=True)
    df_dna['Mutation']=df_dna['Mutation'].str.replace(r"to", "", regex=True)
    df_dna['Mutation']=df_dna['Mutation'].str.replace(r"stop", "*", regex=True)
    return df_dna

def dnaNpro_benchmark_tool_with_gt(
    df_tool_dna: pd.DataFrame,
    df_gt_dna: pd.DataFrame,
    tool: str,
    gt_col: str = "Mutation",
    tool_col: str = "Mutation",
    match_mode: str = "substring",   # "exact" or "substring"
    macro: bool = True,
    verbose: bool = True
):
    """
    Compare tool vs GT for DNA variants using per-PMID sets.

    Returns:
      merged_df: per-PMID sets + TP/FP/FN + per-PMID metrics
      summary: dict of micro + macro metrics

    Metrics:
      precision = TP / (TP + FP)
      recall    = TP / (TP + FN)
      f1        = 2PR/(P+R)
      accuracy  = Jaccard = TP / (TP + FP + FN)   (set-based accuracy)
    """

    tool_mut_col = f"{tool}_Mutations"
    tool_total_col = f"Total_{tool}"

    # ---- group into sets per PMID ----
    gt_grouped = df_gt_dna.groupby("PMID")[gt_col].apply(lambda x: set(x.dropna())).reset_index(name="GT_Mutations")
    tool_grouped = df_tool_dna.groupby("PMID")[tool_col].apply(lambda x: set(x.dropna())).reset_index(name=tool_mut_col)

    gt_grouped["PMID"] = gt_grouped["PMID"].astype(str)
    tool_grouped["PMID"] = tool_grouped["PMID"].astype(str)

    merged = pd.merge(gt_grouped, tool_grouped, on="PMID", how="outer")
    merged["GT_Mutations"] = merged["GT_Mutations"].apply(lambda x: x if isinstance(x, set) else set())
    merged[tool_mut_col] = merged[tool_mut_col].apply(lambda x: x if isinstance(x, set) else set())

    # ---- matching functions ----
    def _hit(gt_item: str, tool_item: str) -> bool:
        if match_mode == "exact":
            return gt_item == tool_item
        # substring mode: treat GT as matched if GT string occurs in tool string OR vice versa
        # (robust when one has extra context)
        return (gt_item in tool_item) or (tool_item in gt_item)

    def _count_tp_fp_fn(gt_set: set, tool_set: set):
        # TP: how many GT items are matched by any tool item
        tp = 0
        for g in gt_set:
            if any(_hit(str(g), str(t)) for t in tool_set):
                tp += 1

        fn = len(gt_set) - tp

        # FP: tool items that do not match any GT item
        fp = 0
        for t in tool_set:
            if not any(_hit(str(g), str(t)) for g in gt_set):
                fp += 1

        return tp, fp, fn

    # ---- compute TP/FP/FN per PMID ----
    merged[["TP", "FP", "FN"]] = merged.apply(
        lambda row: pd.Series(_count_tp_fp_fn(row["GT_Mutations"], row[tool_mut_col])),
        axis=1
    )

    merged["Total_GT"] = merged["GT_Mutations"].apply(len)
    merged[tool_total_col] = merged[tool_mut_col].apply(len)

    # per-PMID metrics
    def _prf(tp, fp, fn):
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2*p*r/(p+r)) if (p+r) else 0.0
        acc = tp / (tp + fp + fn) if (tp + fp + fn) else 0.0  # Jaccard accuracy
        return p, r, f1, acc

    merged[["Precision", "Recall", "F1", "Accuracy"]] = merged.apply(
        lambda row: pd.Series(_prf(row["TP"], row["FP"], row["FN"])),
        axis=1
    )

    # ---- micro aggregation ----
    TP = int(merged["TP"].sum())
    FP = int(merged["FP"].sum())
    FN = int(merged["FN"].sum())

    micro_p, micro_r, micro_f1, micro_acc = _prf(TP, FP, FN)

    summary = {
        "tool": tool,
        "match_mode": match_mode,
        "micro": {
            "TP": TP, "FP": FP, "FN": FN,
            "precision": micro_p,
            "recall": micro_r,
            "f1": micro_f1,
            "accuracy_jaccard": micro_acc,
        }
    }

    # ---- macro aggregation (optional) ----
    if macro:
        df_nonempty = merged[merged["Total_GT"] > 0].copy()
        summary["macro"] = {
            "precision": float(df_nonempty["Precision"].mean()) if len(df_nonempty) else 0.0,
            "recall": float(df_nonempty["Recall"].mean()) if len(df_nonempty) else 0.0,
            "f1": float(df_nonempty["F1"].mean()) if len(df_nonempty) else 0.0,
            "accuracy_jaccard": float(df_nonempty["Accuracy"].mean()) if len(df_nonempty) else 0.0,
        }

    if verbose:
        m = summary["micro"]
        print(
            f"## {tool} DNA benchmark ({match_mode}) | "
            f"TP={m['TP']} FP={m['FP']} FN={m['FN']} | "
            f"P={m['precision']:.4f} R={m['recall']:.4f} F1={m['f1']:.4f} Acc(Jacc)={m['accuracy_jaccard']:.4f}"
        )

    # return per-PMID details + summary
    keep_cols = ["PMID", "GT_Mutations", tool_mut_col, "TP", "FP", "FN", "Total_GT", tool_total_col,
                 "Precision", "Recall", "F1", "Accuracy"]
    return merged[keep_cols], summary


In [20]:
seth_tv3=seth_tv3.rename(columns={'text':"Mutation"})

In [22]:
seth_tv3['type'].unique()

array(['SUBSTITUTION', 'INSERTION', 'DELETION', 'DUPLICATION',
       'DBSNP_MENTION', 'DELETION_INSERTION', 'FRAMESHIFT',
       'STRUCTURAL_ABNORMALITY', 'SHORT_SEQUENCE_REPEAT'], dtype=object)

In [28]:
result_seth, summary_seth = dnaNpro_benchmark_tool_with_gt(seth_tv3,gt_tv3_dnaNpro,'SETH')

## SETH DNA benchmark (substring) | TP=808 FP=4 FN=589 | P=0.9951 R=0.5784 F1=0.7316 Acc(Jacc)=0.5767


In [27]:
result_seth2, summary_seth2 = dnaNpro_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(seth_tv3),
                                                           clean_dnaNpro_mut_for_benchmark(gt_tv3_dnaNpro),
                                                           'SETH')

## SETH DNA benchmark (substring) | TP=793 FP=4 FN=470 | P=0.9950 R=0.6279 F1=0.7699 Acc(Jacc)=0.6259


In [35]:
aioner_tv3 = read_pubtator_format('results/AIONER/tmVar3Corpus_text_only.txt')[[0,3,4]]
aioner_tv3.columns = ['PMID','Mutation','Type']
aioner_tv3

,PMID,Mutation,Type
0,22051099,CXCR1,Gene
1,22051099,IL8RA,Gene
2,22051099,chronic periodontitis,Disease
3,22051099,chemokine receptor 1 CXCR-1,Gene
4,22051099,IL8R-alpha,Gene
...,...,...,...
13415,12626442,MBL,Gene
13416,12626442,MBL,Gene
13417,12626442,C4,Gene
13418,12626442,C4B deficiency,Disease


In [36]:
aioner_tv3['Type'].unique()

array(['Gene', 'Disease', 'Variant', 'Species', 'Chemical', 'CellLine'],
      dtype=object)

In [37]:
aioner_tv3 = aioner_tv3[aioner_tv3['Type']=='Variant']

In [38]:
aioner_tv3

,PMID,Mutation,Type
8,22051099,rs2234671,Variant
9,22051099,Ex2+860G>C,Variant
11,22051099,S276T,Variant
16,22051099,rs2234671,Variant
18,22051099,rs2234671,Variant
...,...,...,...
13374,12650796,cytosine insertion at position 3020,Variant
13375,12650796,3020insC,Variant
13386,12650796,3020insC,Variant
13389,12650796,3020insC,Variant


In [40]:
result_aioner, summary_aioner = dnaNpro_benchmark_tool_with_gt(aioner_tv3,gt_tv3_dnaNpro,'AIONER')

## AIONER DNA benchmark (substring) | TP=1377 FP=23 FN=20 | P=0.9836 R=0.9857 F1=0.9846 Acc(Jacc)=0.9697


In [41]:
result_aioner2, summary_aioner2 = dnaNpro_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(aioner_tv3),
                                                           clean_dnaNpro_mut_for_benchmark(gt_tv3_dnaNpro),
                                                           'AIONER')

## AIONER DNA benchmark (substring) | TP=1246 FP=19 FN=17 | P=0.9850 R=0.9865 F1=0.9858 Acc(Jacc)=0.9719


In [46]:
gt_wei13['Type'].unique()

array(['DNAMutation', 'ProteinMutation', 'SNP'], dtype=object)

In [47]:
gt_wei13

,PMID,Mutation,Type
0,21738389,c.737delC,DNAMutation
1,21738389,p.Pro246HisfsX13,ProteinMutation
2,22051099,rs2234671,SNP
3,22051099,Ex2+860G>C,DNAMutation
4,22051099,S276T,ProteinMutation
...,...,...,...
465,12650796,3020insC,DNAMutation
466,12650796,3020insC,DNAMutation
467,12650796,3020insC,DNAMutation
468,12650796,3020insC,DNAMutation


In [92]:
seth_wei = pd.read_csv('results/SETH/Wei2013_test.seth.out.tsv',sep='\t').rename(columns={'text':'Mutation'})
seth_wei

,PMID,start,end,Mutation,type,tool
0,12650796,897,905,3020insC,INSERTION,REGEX
1,12650796,202,210,3020insC,INSERTION,REGEX
2,12650796,1269,1277,3020insC,INSERTION,REGEX
3,12650796,1063,1071,3020insC,INSERTION,REGEX
4,12829659,172,178,G-308A,SUBSTITUTION,MUTATIONFINDER
...,...,...,...,...,...,...
357,22129472,611,625,c.1706-1715del,DELETION,SETH
358,22129472,835,853,p.(Asp569Valfs*93),FRAMESHIFT,SETH
359,22188495,930,937,3190C>T,SUBSTITUTION,MUTATIONFINDER
360,22188495,1059,1068,Arg987Ter,SUBSTITUTION,MUTATIONFINDER


In [ ]:
aioner_wei = pd.read_csv('results/AIONER/Wei2013_test_text_only.txt')

In [51]:
result_aioner_wei, summary_aioner_wei = dnaNpro_benchmark_tool_with_gt(seth_tv3,gt_wei13,'SETH')
result_aioner2_wei, summary_aioner2_wei = dnaNpro_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(seth_tv3),
                                                           clean_dnaNpro_mut_for_benchmark(gt_wei13),
                                                           'SETH')

## SETH DNA benchmark (substring) | TP=251 FP=530 FN=68 | P=0.3214 R=0.7868 F1=0.4564 Acc(Jacc)=0.2956


In [66]:
result_aioner_wei, summary_aioner_wei = dnaNpro_benchmark_tool_with_gt(aioner_tv3,gt_wei13,'AIONER')
result_aioner2_wei, summary_aioner2_wei = dnaNpro_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(aioner_tv3),
                                                           clean_dnaNpro_mut_for_benchmark(gt_wei13),
                                                           'AIONER')

## AIONER DNA benchmark (substring) | TP=316 FP=954 FN=3 | P=0.2488 R=0.9906 F1=0.3977 Acc(Jacc)=0.2482


In [61]:
#load new gt
gt_seth = pd.read_csv('corpus_groudtruth/SETH-corpus/annotations_combined.tsv',sep='\t',header=0)
gt_seth=gt_seth.rename(columns={'text':'Mutation'})
gt_seth

,PMID,ann_id,entity_type,start,end,Mutation
0,10053017,T1,Gene,1079,1083,YAP+
1,10090486,T1,SNP,341,347,CD6 -G
2,10090486,T2,SNP,349,366,CDs 108 /112-12nt
3,10090486,T3,SNP,368,386,CDs 130/131 + GCCT
4,10090486,T4,Gene,475,486,beta-globin
...,...,...,...,...,...,...
3214,9915968,T2,Gene,30,42,presenilin 1
3215,9973288,T1,Gene,44,49,HRPT2
3216,9973288,T3,Gene,468,475,(HRPT2)
3217,9973288,T4,Gene,932,937,HRPT2


In [62]:
gt_seth['entity_type'].unique()

array(['Gene', 'SNP', 'RS'], dtype=object)

In [64]:
seth_seth = pd.read_csv('results/SETH/SETH.seth.out.tsv',sep='\t')
seth_seth=seth_seth.rename(columns={'text':'Mutation'})
seth_seth

,PMID,start,end,Mutation,type,tool
0,10090909,12,20,2314delG,DELETION,REGEX
1,10094560,57,64,544delG,DELETION,REGEX
2,10094560,119,124,E459K,SUBSTITUTION,MUTATIONFINDER
3,10094560,112,117,R411X,SUBSTITUTION,MUTATIONFINDER
4,10094560,838,843,D289V,SUBSTITUTION,MUTATIONFINDER
...,...,...,...,...,...,...
845,9837820,1168,1172,L28P,SUBSTITUTION,MUTATIONFINDER
846,9837820,1244,1252,964del13,DELETION,REGEX
847,9837820,1039,1047,964del13,DELETION,REGEX
848,9837820,1410,1418,964del13,DELETION,REGEX


In [72]:
aioner_seth = read_pubtator_format('results/AIONER/SETH-corpus.pubtator')
aioner_seth

,PMID,start,end,Mutation,Type
0,10841811,25,42,esophageal cancer,Disease
1,10841811,145,162,esophageal cancer,Disease
2,10841811,256,264,patients,Species
3,10841811,277,294,esophageal cancer,Disease
4,10841811,512,529,esophageal cancer,Disease
...,...,...,...,...,...
9319,22623374,642,644,GD,Disease
9320,22623374,704,709,GCase,Gene
9321,22623374,752,754,GD,Disease
9322,22623374,755,763,patients,Species


In [73]:
aioner_seth['Type'].unique()

array(['Disease', 'Species', 'Chemical', 'Gene', 'Variant', 'CellLine'],
      dtype=object)

In [74]:
aioner_seth=aioner_seth[aioner_seth['Type']=='Variant']

In [75]:
result_seth_seth, summary_seth_seth = dnaNpro_benchmark_tool_with_gt(seth_seth,gt_seth,'SETH')
result_seth2_seth, summary_seth2_seth = dnaNpro_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(seth_seth),
                                                           clean_dnaNpro_mut_for_benchmark(gt_seth),
                                                           'SETH')

## SETH DNA benchmark (substring) | TP=607 FP=34 FN=1204 | P=0.9470 R=0.3352 F1=0.4951 Acc(Jacc)=0.3290
## SETH DNA benchmark (substring) | TP=597 FP=34 FN=712 | P=0.9461 R=0.4561 F1=0.6155 Acc(Jacc)=0.4445


In [76]:
result_aioner_seth, summary_aioner_seth = dnaNpro_benchmark_tool_with_gt(aioner_seth,gt_seth,'AIONER')
result_aioner2_seth, summary_aioner2_seth = dnaNpro_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(aioner_seth),
                                                           clean_dnaNpro_mut_for_benchmark(gt_seth),
                                                           'AIONER')

## AIONER DNA benchmark (substring) | TP=686 FP=148 FN=1125 | P=0.8225 R=0.3788 F1=0.5187 Acc(Jacc)=0.3502
## AIONER DNA benchmark (substring) | TP=663 FP=99 FN=646 | P=0.8701 R=0.5065 F1=0.6403 Acc(Jacc)=0.4709


In [84]:
gt_osiris,_ = load_seth_corpus_xml_labels('corpus_groudtruth/OSIRIS/corpus.xml')
gt_osiris

,PMID,start,end,text,Type,section,source,g_id,g_lex,v_id,v_lex,v_norm
0,12890863,14,46,transforming growth factor beta1,Gene,Title,SETH_XML,7040,transforming growth factor beta1,NaN,NaN,NaN
1,12890863,163,195,transforming growth factor beta1,Gene,Abstract,SETH_XML,7040,transforming growth factor beta1,NaN,NaN,NaN
2,12890863,197,202,TGFB1,Gene,Abstract,SETH_XML,7040,TGFB1,NaN,NaN,NaN
3,12890863,402,407,TGFB1,Gene,Abstract,SETH_XML,7040,TGFB1,NaN,NaN,NaN
4,12890863,1007,1012,TGFB1,Gene,Abstract,SETH_XML,7040,TGFB1,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1360,15170937,1516,1521,IL-1R,Gene,Abstract,SETH_XML,3554,IL-1R,NaN,NaN,NaN
1361,15170937,1521,1527,IL-1RA,Gene,Abstract,SETH_XML,3557,IL-1RA,NaN,NaN,NaN
1362,15170937,1601,1605,IL-4,Gene,Abstract,SETH_XML,3565,IL-4,NaN,NaN,NaN
1363,15170937,1606,1615,-1098 T/G,Variant,Abstract,SETH_XML,NaN,NaN,No,-1098 T/G,T-1098G


In [86]:
gt_osiris = gt_osiris[gt_osiris['Type']=='Variant']
gt_osiris=gt_osiris.rename(columns={'v_lex':'Mutation'})
gt_osiris.head()

,PMID,start,end,text,Type,section,source,g_id,g_lex,v_id,Mutation,v_norm
5,12890863,1013,1020,+1632 T,Variant,Abstract,SETH_XML,NaN,NaN,No,+1632 T,1632T
18,12915397,600,608,Ala55Val,Variant,Abstract,SETH_XML,NaN,NaN,660339,Ala55Val,A55V
19,12915397,612,616,A55V,Variant,Abstract,SETH_XML,NaN,NaN,660339,A55V,A55V
24,12915397,1803,1809,-866 A,Variant,Abstract,SETH_XML,NaN,NaN,No,-866 A,-866A
31,14523377,1088,1097,469+14G/C,Variant,Abstract,SETH_XML,NaN,NaN,3731865,469+14G/C,G469+14C


In [90]:
seth_osiris = pd.read_csv('results/SETH/OSIRIS.seth.out.tsv',sep='\t').rename(columns={'text':'Mutation'})
seth_osiris

,PMID,start,end,Mutation,type,tool
0,12915397,513,521,Ala55Val,SUBSTITUTION,MUTATIONFINDER
1,12915397,525,529,A55V,SUBSTITUTION,MUTATIONFINDER
2,14523377,1056,1065,469+14G/C,SUBSTITUTION,MUTATIONFINDER
3,14523377,1003,1012,469+14G/C,SUBSTITUTION,MUTATIONFINDER
4,14523377,1434,1440,-86G/A,SUBSTITUTION,MUTATIONFINDER
...,...,...,...,...,...,...
328,15166293,594,599,Y181H,SUBSTITUTION,MUTATIONFINDER
329,15166293,505,510,R248Q,SUBSTITUTION,MUTATIONFINDER
330,15166293,430,435,Y181H,SUBSTITUTION,MUTATIONFINDER
331,15170937,960,969,-1098 T/G,SUBSTITUTION,MUTATIONFINDER


In [ ]:
aioner_

In [ ]:
result_seth_osiris, summary_seth_osiris = dnaNpro_benchmark_tool_with_gt(seth_seth,gt_seth,'SETH')
result_seth2_osiris, summary_seth2_osiris = dnaNpro_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(seth_seth),
                                                           clean_dnaNpro_mut_for_benchmark(gt_seth),
                                                           'SETH')

In [ ]:
result_aioner_osiris, summary_aioner_osiris = dnaNpro_benchmark_tool_with_gt(aioner_seth,gt_seth,'AIONER')
result_aioner2_osiris, summary_aioner2_osiris = dnaNpro_benchmark_tool_with_gt(clean_dnaNpro_mut_for_benchmark(aioner_seth),
                                                           clean_dnaNpro_mut_for_benchmark(gt_seth),
                                                           'AIONER')

# SETH result on tmVar3 corpus

In [31]:
summary_seth2

{'tool': 'SETH',
 'match_mode': 'substring',
 'micro': {'TP': 793,
  'FP': 4,
  'FN': 470,
  'precision': 0.9949811794228356,
  'recall': 0.6278701504354711,
  'f1': 0.7699029126213592,
  'accuracy_jaccard': 0.6258879242304657},
 'macro': {'precision': 0.7419786096256684,
  'recall': 0.5680911498671138,
  'f1': 0.6257990571864456,
  'accuracy_jaccard': 0.5678683334678267}}

# SETH - Wei2013 test corpus

In [53]:
summary_aioner2_wei

{'tool': 'SETH',
 'match_mode': 'substring',
 'micro': {'TP': 251,
  'FP': 530,
  'FN': 68,
  'precision': 0.3213828425096031,
  'recall': 0.786833855799373,
  'f1': 0.4563636363636364,
  'accuracy_jaccard': 0.2956419316843345},
 'macro': {'precision': 0.8325082508250826,
  'recall': 0.7503194550224254,
  'f1': 0.7786024785243127,
  'accuracy_jaccard': 0.7412435474316662}}

# AIONER - tmVar3 corpus

In [42]:
summary_aioner2

{'tool': 'AIONER',
 'match_mode': 'substring',
 'micro': {'TP': 1246,
  'FP': 19,
  'FN': 17,
  'precision': 0.9849802371541502,
  'recall': 0.9865399841646872,
  'f1': 0.985759493670886,
  'accuracy_jaccard': 0.9719188767550702},
 'macro': {'precision': 0.990734888996921,
  'recall': 0.984038918597742,
  'f1': 0.9847251556594118,
  'accuracy_jaccard': 0.9774818838321513}}

# AIONER - Wei2013 test corpus

In [52]:
summary_aioner2_wei

{'tool': 'SETH',
 'match_mode': 'substring',
 'micro': {'TP': 251,
  'FP': 530,
  'FN': 68,
  'precision': 0.3213828425096031,
  'recall': 0.786833855799373,
  'f1': 0.4563636363636364,
  'accuracy_jaccard': 0.2956419316843345},
 'macro': {'precision': 0.8325082508250826,
  'recall': 0.7503194550224254,
  'f1': 0.7786024785243127,
  'accuracy_jaccard': 0.7412435474316662}}

In [44]:
gt_wei13

,PMID,Mutation,Type
0,21738389,c.737delC,DNAMutation
1,21738389,p.Pro246HisfsX13,ProteinMutation
2,22051099,rs2234671,SNP
3,22051099,Ex2+860G>C,DNAMutation
4,22051099,S276T,ProteinMutation
...,...,...,...
465,12650796,3020insC,DNAMutation
466,12650796,3020insC,DNAMutation
467,12650796,3020insC,DNAMutation
468,12650796,3020insC,DNAMutation
